In [59]:
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage , SystemMessage
from langgraph.graph import StateGraph,START,END
from typing import Literal,List, Annotated, TypedDict
from pydantic import BaseModel, Field


# from langchain_groq import ChatGroq
# from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [47]:
generative_llm =ChatGroq(model='llama-3.1-8b-instant',api_key='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')
# evaluate_llm = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
# optimizer_llm  = ChatGroq(model="llama-3.1-8b-instant",api_key="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")


In [71]:
class TweetEvaluation(BaseModel):
    evaluation :Literal["approved", "needs_approved"]
    feedback : str=Field(... , description="feedback for the tweet")

In [73]:
# state
class Tweet_state(TypedDict):
    topic:str
    tweet:str
    evaluation : Literal["approved", "needs_approved"]
    feedback :str
    iteration : int
    max_iteration : int

In [79]:
structure_evaluate_llm = generative_llm.with_structured_output(TweetEvaluation)

In [75]:
def generate_tweet(state: Tweet_state):
    message = [
        SystemMessage(content="you are a so funny and clever/X influencer"),
        HumanMessage(content=f"""
        write a short , original, and hillarious tweet on this topic {state['topic']}
Rules:
 - Do not question answere format.
 - max 280 character.
 - use observational humor , irony, sarscam on cultural refrence
 - Think in meme logic, punclines , or relatable takes
 - use simple , day to day english """)]
# genetaor llm
    response = generative_llm.invoke(message)

#  add this 
    return {"tweet": response}

In [81]:
def evaluate_tweet(state: Tweet_state):
    message = [
        SystemMessage(content="You are a witty social media expert and content critic."),
        HumanMessage(content=f"""
You are given a tweet and its topic. Evaluate the tweet based on the following criteria:
1. Humor: Is it genuinely funny, clever, and shareable?
2. Relevance: Does it relate directly to the topic '{state['topic']}'?
3. Clarity: Is it easy to read and understand?
4. Engagement: Is it likely to get likes, retweets, or comments?
5. Tone: Does it match a witty, sarcastic, or meme-like style suitable for X/Twitter?

Tweet:
"{state['tweet']}"

Instructions:
- Give a short comment for each point above.
- Provide an overall assessment as either "approved" or "needs_approved".
- Suggest at least one improvement or tip if it "needs_approved".
- Do NOT rewrite the tweet yet.
- Keep the response concise and structured like:

Humor: 8/10, funny and clever  
Relevance: 9/10, matches topic  
Clarity: 7/10, slightly long  
Engagement: 8/10, likely to get likes  
Tone: 9/10, fits social media style  
Overall: approved  
Feedback: Great tweet, minor edits could make it sharper
""")]
    response = structure_evaluate_llm.invoke(message)
    return {"evaluation":response.evaluation, "feedback": response.feedback }
   

In [83]:
def optimize_tweet(state: Tweet_state):
    message = [
        SystemMessage(content="You are a clever social media content optimizer."),
        HumanMessage(content=f"""
The tweet below received feedback: "{state['feedback']}".
Topic: {state['topic']}
Tweet: "{state['tweet']}"

Instructions:
- Improve the tweet based on the feedback.
- Keep it funny, clever, and relevant to the topic.
- Max 280 characters.
- Use observational humor, sarcasm, or meme-style punchlines.
- Do not remove the essence of the original tweet.
""")
    ]
    response = generative_llm.invoke(message).content
    iteration = state['iteration'] + 1
    return {"tweet":response, "iteration":iteration}

In [91]:
def route_evaluate(state: Tweet_state):
    if state['evaluation'] == "approved" or state["iteration"] >= state["max_iteration"]:
        return "approved"
    else:
        return "needs_approved"

In [93]:
graph = StateGraph(Tweet_state)
# add nodes
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

#  add adges

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_evaluate, {"approved":END, "needs_approved": "optimize"})
graph.add_edge("optimize","evaluate")


In [97]:
workflow = graph.compile()

In [125]:
initial_state = {
    "topic": "you are  a bad man",
    "iteration" : 1,
    "max_iteration" : 5
}
workflow.invoke(initial_state)

{'topic': 'you are  a bad man',
 'tweet': AIMessage(content='"I\'m a bad man, but I\'m also a good liar, a great cook (with microwaved meals), and a decent Spotify curator. If I had a nickel for every bad decision, I\'d have enough to buy a decent therapist #BadManVibes"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 122, 'total_tokens': 179, 'completion_time': 0.130191254, 'completion_tokens_details': None, 'prompt_time': 0.010467629, 'prompt_tokens_details': None, 'queue_time': 0.005657006, 'total_time': 0.140658883}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b0288-2db6-7363-b991-0e50106e28ac-0', usage_metadata={'input_tokens': 122, 'output_tokens': 57, 'total_tokens': 179}),
 'evaluation': 'approved',
 'feedback': 'Great tweet, minor edits could make it sharper',
 'iteration': 1,
 'm